In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
load_dotenv(override=True)

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [4]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local(
    "./faiss_index",
    embeddings,
    allow_dangerous_deserialization=True # 데이터 역직렬화 허용
)

In [6]:
retriever = vectorstore.as_retriever()

In [9]:
query = "결혼하면 얼마를 받을 수 있을까?"

In [13]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """다음 컨텍스트만 사용해 질문에 답하세요.
컨텍스트: {context}

질문: {question}
"""
)

In [17]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [27]:
from langchain_core.runnables import RunnablePassthrough
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash-lite")

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | parser
)

In [28]:
res = chain.invoke(query)

In [29]:
res

'결혼 시 받을 수 있는 금액은 다음과 같습니다.\n\n*   **본인 결혼:** 100만원 + 화환 지원\n*   **자녀 결혼:** 50만원 + 화환 지원\n\n비혼 선언을 한 경우, 결혼 축의금과 동일한 수준의 보상이 제공되며 비혼 축하금으로 100만원이 지원됩니다.'

In [33]:
query = "자격증 비용은 얼마를 받을 수 있어?"

In [34]:
res = chain.invoke(query)

In [35]:
res

'직무 관련 국가 기술 자격 취득 시 다음과 같은 축하금 및 수당을 받을 수 있습니다.\n\n*   **기술사/기능장**: 축하금 200만원, 월 30만원\n*   **기사**: 축하금 50만원, 월 10만원\n*   **산업기사**: 축하금 30만원, 월 5만원\n*   **기능사**: 축하금 10만원, 월 3만원\n\n축하금은 1회성으로 지급되며, 수당은 자격증 사본을 HR 팀에 제출한 익월 급여부터 반영됩니다. 동일 등급 내 1개 자격증만 수당으로 인정되며, 상위 등급 취득 시 갱신됩니다. 축하금은 횟수 제한이 없습니다.'